# MCH–MFT Track Matching — Training & Testing

Regular (XGBoost Logistic) to score (MCH, MFT candidate) pairs.  
Each MCH track forms a **group** of up to 5 candidates; the model learns to rank the true match highest within each group.
The top-ranked candidate is accepted if its score exceeds a tunable threshold.

**Stages:** TODO: amend
1. Load data with hipe4ml
2. Feature engineering
3. Train/test split (by MCH track group)
4. XGBoost LambdaRank training
5. Group-level evaluation (efficiency, purity, MRR)

## 0. Imports

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import xgboost as xgb
import importlib
import Utils

from hipe4ml.tree_handler import TreeHandler
from sklearn.model_selection import GroupShuffleSplit
from scipy.special import softmax

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

## 1. Load, Format, Engineer Data

In [ ]:
df_original = Utils.get_dataframe("FwdMatchMLCandidatesFull.root")


In [ ]:
df =df_original.copy() 
importlib.reload(Utils)

In [ ]:
df = Utils.process_dataframe(df, makedummies=False)

# FEATURES = [f for f in df.columns.tolist() if f not in Utils.NON_TRAINING_FEATURES]
FEATURES = [
    "DeltaDirection",
    "CPhiPhiMFT",
    "PtMFT",
    "RelPtDiff",
    "CXXMFT",
    "DeltaPt",
    "PullTanl",
    "DCAX",
    "DCAY",
    "SameSign",
    "PullPhi",
    "DeltaR",
]


TARGET = "IsSignal"

GROUP  = "mchID"

In [ ]:
df[FEATURES].describe()

In [ ]:
FEATURES

## 2. Sanity Checks

In [ ]:
n_mch_tracks = df["mchID"].nunique()
n_positive = df["IsSignal"].sum()
candidates_per_track = df.groupby("mchID").size()

print(f"MCH tracks:          {n_mch_tracks:,}")
print(f"Total pairs:         {len(df):,}")
print(f"True matches:        {int(n_positive):,} ({100*n_positive/len(df):.1f}%)")
print(f"Candidates per track: min={candidates_per_track.min()}, "
      f"max={candidates_per_track.max()}, "
      f"mean={candidates_per_track.mean():.2f}")

# Tracks with no true match among candidates
tracks_with_match = df.groupby("mchID")["IsSignal"].max()
n_no_match = (tracks_with_match == 0).sum()
print(f"Tracks with no true match in candidates: {n_no_match:,} "
      f"({100*n_no_match/n_mch_tracks:.1f}%)")

## 4. Train / Test Split

Split is done **by MCH track group**, not by row, to avoid data leakage  
(candidates from the same MCH track must not appear in both train and test).

In [ ]:
groups   = df[GROUP].values
splitter = GroupShuffleSplit(n_splits=1, test_size=0.3, random_state=42) # shuffle the data and split into train/test sets while ensuring all candidates from the same mchID are in the same set
train_idx, test_idx = next(splitter.split(df, groups=groups))

df_train = df.iloc[train_idx].copy()
df_test  = df.iloc[test_idx].copy()

X_train, y_train = df_train[FEATURES], df_train[TARGET]
X_test,  y_test  = df_test[FEATURES],  df_test[TARGET]

print(f"Train: {len(df_train):,} pairs ({df_train[GROUP].nunique():,} MCH tracks)")
print(f"Test:  {len(df_test):,} pairs ({df_test[GROUP].nunique():,} MCH tracks)")
print(f"Train positive rate: {y_train.mean():.3f}")
print(f"Test  positive rate: {y_test.mean():.3f}")

## 5. Train XGBoost LambdaRank

`objective='rank:ndcg'` optimises the ranking within each group of candidates directly,  
avoiding the class-imbalance problem of a binary classifier and matching the problem structure exactly.  

In [ ]:
neg = (y_train == 0).sum()
pos = (y_train == 1).sum()
spw = neg / pos
print(f"scale_pos_weight = {spw:.2f}  (neg={neg:,}, pos={pos:,})")

model = xgb.XGBClassifier(
    objective="binary:logistic",
    n_estimators=500,
    max_depth=8,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=spw,
    eval_metric="auc",
    early_stopping_rounds=20,
    random_state=42,
    n_jobs=-1,
)

model.fit(
    X_train, y_train,
    eval_set=[(X_train, y_train), (X_test, y_test)],
    verbose=50,
)

print(f"\nBest iteration: {model.best_iteration}")

# ── Training curve ────────────────────────────────────────────────────────────
results   = model.evals_result()
train_auc = results["validation_0"]["auc"]
test_auc  = results["validation_1"]["auc"]
iterations = range(1, len(train_auc) + 1)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left — full training curve
ax = axes[0]
ax.plot(iterations, train_auc, lw=2, color="steelblue", label="Train AUC")
ax.plot(iterations, test_auc,  lw=2, color="tomato",    label="Test AUC")
ax.axvline(model.best_iteration, color="black", linestyle="--", lw=1.5,
           label=f"Best iteration ({model.best_iteration})")
ax.set_xlabel("Iteration")
ax.set_ylabel("AUC")
ax.set_title("Training curve — full")
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)

# Right — zoomed to last 20%
zoom_start = int(0.8 * len(train_auc))
ax = axes[1]
ax.plot(list(iterations)[zoom_start:], train_auc[zoom_start:],
        lw=2, color="steelblue", label="Train AUC")
ax.plot(list(iterations)[zoom_start:], test_auc[zoom_start:],
        lw=2, color="tomato",    label="Test AUC")
ax.axvline(model.best_iteration, color="black", linestyle="--", lw=1.5,
           label=f"Best iteration ({model.best_iteration})")
ax.set_xlabel("Iteration")
ax.set_ylabel("AUC")
ax.set_title("Training curve — zoomed (last 20%)")
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

# ── Gap diagnostic ────────────────────────────────────────────────────────────
final_gap = abs(train_auc[-1] - test_auc[-1])
best_gap  = abs(train_auc[model.best_iteration - 1] -
                test_auc[model.best_iteration - 1])
print(f"Train/test AUC gap at best iteration: {best_gap:.6f}")
print(f"Train/test AUC gap at final iteration: {final_gap:.6f}")
print(f"{'⚠ Possible overfitting' if final_gap > 0.01 else '✓ No significant overfitting detected'}")

## 6. Feature Importance

In [ ]:
importances = pd.Series(
    model.feature_importances_, index=FEATURES
).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 10))
importances.plot.barh(ax=ax)
ax.set_xlabel("Feature importance (gain)")
ax.set_title("XGBoost feature importances")
plt.tight_layout()
plt.show()

## 7. Group-Level Evaluation

In [ ]:
df_test = df_test.copy()
df_test["score"] = model.predict_proba(X_test)[:, 1]


METRICS = ["score"]

In [ ]:
g = xgb.to_graphviz(
    model,
    num_trees=10,
    graph_attr={"dpi": "300", "size": "20,10"}
)

g.render("xgb_tree", format="png", cleanup=True)

## 8. All matches

In [ ]:
match_groups = Utils.build_match_groups(df_test)
Utils.draw_all_features(features=METRICS, match_groups=match_groups, density=True)

## 9. Leading Match Metric Distribution Analysis

In [ ]:
importlib.reload(Utils)
df_leader = df_test.loc[df_test.groupby("mchID")["score"].idxmax()].reset_index(drop=True)
match_groups_leader = Utils.build_match_groups(df_leader)
Utils.draw_all_features(features=METRICS, match_groups=match_groups_leader, density=True, per=0.0)

## 10. Metric score sweep

In [ ]:
for entry in METRICS:
    Utils.sweep_threshold_plot(df_eval= df_test, metrics_fn = Utils.inhousemetrics, title=entry +" vs Score Threshold", score_col=entry, Nsigma=3.0)

## 11. Match Assigned Analysis

In [ ]:
# Apply threshold to get final matches, then plot feature distributions for the accepted candidates - see firsthand the contamination
threshold = 0.6
df_leader = df_test.loc[df_test.groupby("mchID")["score"].idxmax()].reset_index(drop=True)
df_leader = df_leader[df_leader["score"] >= threshold].reset_index(drop=True)
match_groups_leader = Utils.build_match_groups(df_leader)
Utils.draw_all_features(features=FEATURES, match_groups=match_groups_leader, density=True)

## 12. Featurewise Metric breakdown

In [ ]:
importlib.reload(Utils)
for entry in Utils.DESIGNED_FEATURES:   
    Utils.plot_metrics_vs_feature(df=df_test,feature=entry, threshold = 0.6, metrics_fn=Utils.inhousemetrics, metric_col_prefix="score", bins=25, trim_low=0.02, trim_high=0.02, Nsigma=2.0)

In [ ]:
df_debug = Utils.plot_metrics_vs_feature(df=df_test,feature='PtMCH', threshold = 0.5, metrics_fn=Utils.inhousemetrics, metric_col_prefix="score_softmax", bins=25, trim_low=0.02, trim_high=0.02, Nsigma=1.0)

## 13. Feature 

In [ ]:
from scipy.stats import ks_2samp

# ── Thresholds — adjust these ─────────────────────────────────────────────────
KS_THRESHOLD         = 0.1
IMPORTANCE_THRESHOLD = 0.01

# ── Compute KS statistic for each feature ────────────────────────────────────
signal     = df[df["IsSignal"] == 1]
background = df[df["IsSignal"] == 0]

ks_stats = {
    feature: ks_2samp(signal[feature].dropna(),
                      background[feature].dropna()).statistic
    for feature in FEATURES
}

importance = dict(zip(FEATURES, model.feature_importances_))

selection_df = pd.DataFrame({
    "ks":         ks_stats,
    "importance": importance,
}).sort_values("ks", ascending=False)

# ── Plot ──────────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(9, 6))

passes = selection_df[
    (selection_df["ks"]         >= KS_THRESHOLD) &
    (selection_df["importance"] >= IMPORTANCE_THRESHOLD)
]
fails = selection_df[
    (selection_df["ks"]         <  KS_THRESHOLD) |
    (selection_df["importance"] <  IMPORTANCE_THRESHOLD)
]

ax.scatter(fails["ks"],   fails["importance"],
           alpha=0.6, color="tomato",    s=60, label="Rejected")
ax.scatter(passes["ks"],  passes["importance"],
           alpha=0.8, color="steelblue", s=60, label="Selected")

for feature, row in selection_df.iterrows():
    ax.annotate(feature, (row["ks"], row["importance"]),
                fontsize=8, textcoords="offset points", xytext=(5, 3))

ax.axvline(KS_THRESHOLD,         color="black", linestyle="--", lw=1.2,
           label=f"KS threshold ({KS_THRESHOLD})")
ax.axhline(IMPORTANCE_THRESHOLD, color="grey",  linestyle="--", lw=1.2,
           label=f"Importance threshold ({IMPORTANCE_THRESHOLD})")

ax.set_xlabel("KS statistic (univariate separability)")
ax.set_ylabel("Feature importance (model usage)")
ax.set_title("Feature selection overview")
ax.legend(frameon=False)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── Print selected features ───────────────────────────────────────────────────
selected = passes.index.tolist()
dropped  = fails.index.tolist()

print(f"Selected ({len(selected)}) — copy-paste ready:")
print("FEATURES = [")
for f in selected:
    print(f"    \"{f}\",")
print("]")

print(f"\nDropped ({len(dropped)}):")
print(dropped)
print(selection_df.round(4))

## 14. ONNX

In [ ]:
print(xgb.__version__)

In [ ]:
from hipe4ml.model_handler import ModelHandler
import onnxmltools
from onnxmltools.convert.common.data_types import FloatTensorType
import json


In [ ]:
# --- XGBoost → ONNX (patched for XGBoost ≥ 2.x/3.x) ---

# 1. Patch base_score issue (XGBoost >= 2.0 compatibility)
booster = model.get_booster()
config = json.loads(booster.save_config())

bs = config["learner"]["learner_model_param"].get("base_score", None)

if isinstance(bs, str) and bs.startswith("["):
    try:
        config["learner"]["learner_model_param"]["base_score"] = float(bs.strip("[]"))
        print(f"[INFO] Patched base_score from {bs} → {config['learner']['learner_model_param']['base_score']}")
    except Exception as e:
        raise RuntimeError(f"Failed to patch base_score: {bs}") from e

# Apply patched config
booster.load_config(json.dumps(config))

# 2. Define ONNX input type
n_features = X_train.shape[1]
initial_type = [('input', FloatTensorType([None, n_features]))]

# 3. Convert
onnx_model = onnxmltools.convert_xgboost(model, initial_types=initial_type)

# 4. Save
onnx_path = "model.onnx"
with open(onnx_path, "wb") as f:
    f.write(onnx_model.SerializeToString())

print(f"[SUCCESS] ONNX model saved to: {onnx_path}")